# TF-IDF Vectorization and Cosine Similarity Training
## Medicine Name Matching and Similarity Analysis

This notebook demonstrates:
1. TF-IDF vectorization for text processing
2. Cosine similarity calculations
3. Medicine name matching between extracted text and database
4. Vector comparison visualization
5. Practical applications for OCR-extracted medicine names

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 1. Create Sample Medicine Database

First, let's create a sample database of medicine names that might exist in our system.

In [ ]:
# Sample medicine database (simulating a real pharmaceutical database)
medicine_database = [
    "Paracetamol", "Acetaminophen", "Ibuprofen", "Aspirin", "Diclofenac",
    "Amoxicillin", "Azithromycin", "Ciprofloxacin", "Metformin", "Insulin",
    "Atorvastatin", "Simvastatin", "Amlodipine", "Losartan", "Omeprazole",
    "Pantoprazole", "Cetirizine", "Loratadine", "Prednisolone", "Hydrocortisone",
    "Salbutamol", "Albuterol", "Montelukast", "Warfarin", "Clopidogrel",
    "Furosemide", "Hydrochlorothiazide", "Ramipril", "Enalapril", "Bisoprolol"
]

print(f"Medicine Database Created with {len(medicine_database)} medicines:")
for i, med in enumerate(medicine_database[:10], 1):
    print(f"{i:2d}. {med}")
print("... and 20 more medicines")

## 2. Simulate OCR Extracted Text

Let's simulate some medicine names that might be extracted from images using OCR, including some with errors or variations.

In [ ]:
# Simulated OCR extracted text (with potential errors and variations)
ocr_extracted_text = [
    "Paracetamol",      # Exact match
    "Paracetamo1",      # OCR error (l -> 1)
    "IBUPROFEN",        # Case variation
    "lbuprofen",        # OCR error (I -> l)
    "Asprin",           # Missing letter
    "Amoxicilin",       # Missing letter
    "Azithromycin",     # Exact match
    "Ciprofloxacin",    # Exact match
    "Metfornin",        # OCR error (m -> rn)
    "lnsulin",          # OCR error (I -> l)
    "Atorvastatin",     # Exact match
    "Simvastatin",      # Exact match
    "Am1odipine",       # OCR error (l -> 1)
    "Losarten",         # OCR error (a -> e)
    "0meprazole",       # OCR error (O -> 0)
    "Pantoprazo1e",     # OCR error (l -> 1)
    "Cetirizine",       # Exact match
    "Loratadine",       # Exact match
    "Pred nisolone",    # Space insertion
    "Hydrocortisone"     # Exact match
]

print("OCR Extracted Medicine Names (with potential errors):")
for i, text in enumerate(ocr_extracted_text, 1):
    print(f"{i:2d}. '{text}'")

## 3. Text Preprocessing

Before vectorization, let's implement some text preprocessing to handle common OCR errors and variations.

In [ ]:
import re
import string

def preprocess_text(text):
    """
    Preprocess text for better matching:
    - Convert to lowercase
    - Remove extra spaces
    - Handle common OCR errors
    """
    # Convert to lowercase
    text = text.lower().strip()
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    
    # Handle common OCR errors
    ocr_corrections = {
        '0': 'o',  # Zero to O
        '1': 'l',  # One to l (in some contexts)
        'rn': 'm', # rn combination often misread as m
    }
    
    # Apply corrections (basic approach)
    for error, correction in ocr_corrections.items():
        text = text.replace(error, correction)
    
    return text

# Preprocess all texts
preprocessed_database = [preprocess_text(med) for med in medicine_database]
preprocessed_ocr = [preprocess_text(text) for text in ocr_extracted_text]

print("Original vs Preprocessed OCR Text (first 10):")
for i in range(10):
    print(f"{i+1:2d}. '{ocr_extracted_text[i]}' -> '{preprocessed_ocr[i]}'")

## 4. TF-IDF Vectorization

Now let's create TF-IDF vectors for both the database and extracted text.

In [ ]:
# Combine all texts for vectorization
all_texts = preprocessed_database + preprocessed_ocr

# Initialize TF-IDF Vectorizer
# Using character n-grams (2-4) to handle OCR errors better
tfidf_vectorizer = TfidfVectorizer(
    analyzer='char',        # Character-level analysis
    ngram_range=(2, 4),     # 2-4 character n-grams
    lowercase=True,
    max_features=1000       # Limit features for efficiency
)

# Fit and transform all texts
tfidf_matrix = tfidf_vectorizer.fit_transform(all_texts)

# Split back into database and OCR vectors
database_vectors = tfidf_matrix[:len(preprocessed_database)]
ocr_vectors = tfidf_matrix[len(preprocessed_database):]

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"Number of features (n-grams): {len(tfidf_vectorizer.get_feature_names_out())}")
print(f"Database vectors shape: {database_vectors.shape}")
print(f"OCR vectors shape: {ocr_vectors.shape}")

# Show some example features
print("\nExample character n-grams (features):")
features = tfidf_vectorizer.get_feature_names_out()
print(features[:20])

## 5. Cosine Similarity Calculation

Calculate cosine similarity between each OCR extracted text and all medicines in the database.

In [ ]:
# Calculate cosine similarity matrix
similarity_matrix = cosine_similarity(ocr_vectors, database_vectors)

print(f"Similarity Matrix Shape: {similarity_matrix.shape}")
print(f"Each row represents an OCR text, each column represents a database medicine")

# Function to find best matches
def find_best_matches(ocr_index, top_k=3):
    """
    Find top k best matches for a given OCR extracted text
    """
    similarities = similarity_matrix[ocr_index]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    matches = []
    for idx in top_indices:
        matches.append({
            'medicine': medicine_database[idx],
            'similarity': similarities[idx],
            'preprocessed': preprocessed_database[idx]
        })
    
    return matches

# Demonstrate matching for first 5 OCR texts
print("\nTop 3 matches for each OCR extracted text:")
print("=" * 60)

for i in range(5):
    print(f"\nOCR Text: '{ocr_extracted_text[i]}' (preprocessed: '{preprocessed_ocr[i]}')")
    matches = find_best_matches(i, top_k=3)
    
    for j, match in enumerate(matches, 1):
        print(f"  {j}. {match['medicine']} (similarity: {match['similarity']:.4f})")

## 6. Visualization of Similarity Matrix

Let's visualize the similarity matrix to understand the patterns better.

In [ ]:
# Create a heatmap of similarity scores
plt.figure(figsize=(15, 10))

# Select first 15 OCR texts and 20 database medicines for better visualization
subset_similarity = similarity_matrix[:15, :20]

# Create heatmap
sns.heatmap(
    subset_similarity,
    xticklabels=[med[:15] + '...' if len(med) > 15 else med for med in medicine_database[:20]],
    yticklabels=[text[:15] + '...' if len(text) > 15 else text for text in ocr_extracted_text[:15]],
    annot=True,
    fmt='.3f',
    cmap='YlOrRd',
    cbar_kws={'label': 'Cosine Similarity'}
)

plt.title('Cosine Similarity Matrix: OCR Text vs Database Medicines', fontsize=14, pad=20)
plt.xlabel('Database Medicines', fontsize=12)
plt.ylabel('OCR Extracted Text', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("Heatmap shows similarity scores between OCR texts (rows) and database medicines (columns)")
print("Darker colors indicate higher similarity scores")

## 7. Vector Analysis and Comparison

Let's analyze the actual vectors to understand how TF-IDF captures the similarity.

In [ ]:
# Function to analyze vector differences
def analyze_vectors(ocr_index, db_index):
    """
    Analyze and compare two vectors in detail
    """
    ocr_vec = ocr_vectors[ocr_index].toarray().flatten()
    db_vec = database_vectors[db_index].toarray().flatten()
    
    # Find non-zero features
    ocr_features = np.where(ocr_vec > 0)[0]
    db_features = np.where(db_vec > 0)[0]
    
    # Common features
    common_features = np.intersect1d(ocr_features, db_features)
    
    # Feature names
    feature_names = tfidf_vectorizer.get_feature_names_out()
    
    print(f"OCR Text: '{ocr_extracted_text[ocr_index]}'")
    print(f"Database Medicine: '{medicine_database[db_index]}'")
    print(f"Cosine Similarity: {cosine_similarity([ocr_vec], [db_vec])[0][0]:.4f}")
    print(f"\nVector Analysis:")
    print(f"  OCR features: {len(ocr_features)}")
    print(f"  Database features: {len(db_features)}")
    print(f"  Common features: {len(common_features)}")
    
    if len(common_features) > 0:
        print(f"\nTop common n-grams:")
        # Sort common features by combined TF-IDF scores
        common_scores = ocr_vec[common_features] + db_vec[common_features]
        top_common = common_features[np.argsort(common_scores)[::-1][:10]]
        
        for feat_idx in top_common:
            ngram = feature_names[feat_idx]
            ocr_score = ocr_vec[feat_idx]
            db_score = db_vec[feat_idx]
            print(f"  '{ngram}': OCR={ocr_score:.4f}, DB={db_score:.4f}")

# Analyze a few examples
print("Vector Analysis Examples:")
print("=" * 50)

# Example 1: Exact match
print("\n1. Exact Match Example:")
print("-" * 30)
analyze_vectors(0, 0)  # Paracetamol vs Paracetamol

# Example 2: OCR error
print("\n\n2. OCR Error Example:")
print("-" * 30)
analyze_vectors(1, 0)  # Paracetamo1 vs Paracetamol

## 8. Comprehensive Matching Results

Let's create a comprehensive analysis of all OCR texts and their best matches.

In [ ]:
# Create comprehensive results DataFrame
results = []

for i, ocr_text in enumerate(ocr_extracted_text):
    matches = find_best_matches(i, top_k=1)  # Get best match
    best_match = matches[0]
    
    results.append({
        'OCR_Text': ocr_text,
        'Preprocessed_OCR': preprocessed_ocr[i],
        'Best_Match': best_match['medicine'],
        'Similarity_Score': best_match['similarity'],
        'Match_Quality': 'Exact' if best_match['similarity'] > 0.99 else 
                       'High' if best_match['similarity'] > 0.8 else
                       'Medium' if best_match['similarity'] > 0.6 else
                       'Low'
    })

results_df = pd.DataFrame(results)

# Display results
print("Comprehensive Matching Results:")
print("=" * 80)
print(results_df.to_string(index=False))

# Summary statistics
print("\n\nSummary Statistics:")
print("=" * 30)
print(f"Total OCR texts processed: {len(results_df)}")
print(f"Average similarity score: {results_df['Similarity_Score'].mean():.4f}")
print(f"\nMatch Quality Distribution:")
print(results_df['Match_Quality'].value_counts())

## 9. Performance Analysis and Threshold Setting

Let's analyze different similarity thresholds for practical applications.

In [ ]:
# Analyze different thresholds
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
threshold_analysis = []

for threshold in thresholds:
    accepted = results_df[results_df['Similarity_Score'] >= threshold]
    rejected = results_df[results_df['Similarity_Score'] < threshold]
    
    threshold_analysis.append({
        'Threshold': threshold,
        'Accepted': len(accepted),
        'Rejected': len(rejected),
        'Acceptance_Rate': len(accepted) / len(results_df) * 100,
        'Avg_Score_Accepted': accepted['Similarity_Score'].mean() if len(accepted) > 0 else 0
    })

threshold_df = pd.DataFrame(threshold_analysis)

# Visualize threshold analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Acceptance rate vs threshold
ax1.plot(threshold_df['Threshold'], threshold_df['Acceptance_Rate'], 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Similarity Threshold', fontsize=12)
ax1.set_ylabel('Acceptance Rate (%)', fontsize=12)
ax1.set_title('Acceptance Rate vs Similarity Threshold', fontsize=14)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 105)

# Plot 2: Distribution of similarity scores
ax2.hist(results_df['Similarity_Score'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
ax2.axvline(results_df['Similarity_Score'].mean(), color='red', linestyle='--', 
           label=f'Mean: {results_df["Similarity_Score"].mean():.3f}')
ax2.axvline(results_df['Similarity_Score'].median(), color='green', linestyle='--', 
           label=f'Median: {results_df["Similarity_Score"].median():.3f}')
ax2.set_xlabel('Similarity Score', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Distribution of Similarity Scores', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Threshold Analysis:")
print(threshold_df.to_string(index=False))

# Recommend optimal threshold
print("\nRecommended Thresholds:")
print("- High Confidence: 0.8+ (for automatic matching)")
print("- Medium Confidence: 0.6-0.8 (for manual review)")
print("- Low Confidence: <0.6 (likely no match or manual entry needed)")

## 10. Practical Implementation Function

Let's create a practical function that can be used in the main medicine extraction system.

In [ ]:
class MedicineNameMatcher:
    """
    A practical medicine name matcher using TF-IDF and cosine similarity
    """
    
    def __init__(self, medicine_database, similarity_threshold=0.7):
        self.medicine_database = medicine_database
        self.similarity_threshold = similarity_threshold
        self.preprocessed_database = [self.preprocess_text(med) for med in medicine_database]
        
        # Initialize and fit the vectorizer
        self.vectorizer = TfidfVectorizer(
            analyzer='char',
            ngram_range=(2, 4),
            lowercase=True,
            max_features=1000
        )
        
        # Fit on database
        self.database_vectors = self.vectorizer.fit_transform(self.preprocessed_database)
        
    def preprocess_text(self, text):
        """Preprocess text for better matching"""
        text = text.lower().strip()
        text = re.sub(r'\s+', ' ', text)
        
        # OCR corrections
        ocr_corrections = {'0': 'o', '1': 'l', 'rn': 'm'}
        for error, correction in ocr_corrections.items():
            text = text.replace(error, correction)
            
        return text
    
    def find_best_match(self, ocr_text, top_k=3):
        """
        Find the best matching medicine(s) for the given OCR text
        
        Returns:
            dict: Contains best matches with confidence levels
        """
        # Preprocess input
        preprocessed_input = self.preprocess_text(ocr_text)
        
        # Vectorize input
        input_vector = self.vectorizer.transform([preprocessed_input])
        
        # Calculate similarities
        similarities = cosine_similarity(input_vector, self.database_vectors).flatten()
        
        # Get top matches
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        matches = []
        for idx in top_indices:
            similarity = similarities[idx]
            confidence = 'High' if similarity >= 0.8 else 'Medium' if similarity >= 0.6 else 'Low'
            
            matches.append({
                'medicine': self.medicine_database[idx],
                'similarity': similarity,
                'confidence': confidence,
                'above_threshold': similarity >= self.similarity_threshold
            })
        
        return {
            'input_text': ocr_text,
            'preprocessed_input': preprocessed_input,
            'matches': matches,
            'best_match': matches[0] if matches else None,
            'recommended_action': self._get_recommendation(matches[0] if matches else None)
        }
    
    def _get_recommendation(self, best_match):
        """Get recommendation based on best match"""
        if not best_match:
            return "No match found - manual entry required"
        
        if best_match['similarity'] >= 0.9:
            return "Auto-accept - very high confidence"
        elif best_match['similarity'] >= 0.8:
            return "Auto-accept - high confidence"
        elif best_match['similarity'] >= 0.6:
            return "Manual review recommended - medium confidence"
        else:
            return "Manual entry likely needed - low confidence"

# Test the matcher
matcher = MedicineNameMatcher(medicine_database, similarity_threshold=0.7)

print("Testing the Medicine Name Matcher:")
print("=" * 50)

test_cases = ["Paracetamo1", "IBUPROFEN", "Asprin", "UnknownMedicine", "lnsulin"]

for test_case in test_cases:
    result = matcher.find_best_match(test_case, top_k=2)
    
    print(f"\nInput: '{test_case}'")
    print(f"Best Match: {result['best_match']['medicine']} "
          f"(similarity: {result['best_match']['similarity']:.4f})")
    print(f"Confidence: {result['best_match']['confidence']}")
    print(f"Recommendation: {result['recommended_action']}")

## 11. Integration with Medicine Extraction System

Here's how this can be integrated with your existing medicine extraction system.

In [ ]:
# Example integration function
def process_extracted_medicines(extracted_texts, medicine_database):
    """
    Process a batch of OCR-extracted medicine names
    
    Args:
        extracted_texts: List of medicine names extracted from images
        medicine_database: List of known medicine names
    
    Returns:
        dict: Processing results with matches and recommendations
    """
    matcher = MedicineNameMatcher(medicine_database)
    
    results = {
        'processed_medicines': [],
        'auto_accepted': [],
        'manual_review': [],
        'manual_entry': [],
        'statistics': {}
    }
    
    for text in extracted_texts:
        match_result = matcher.find_best_match(text)
        
        # Categorize based on recommendation
        recommendation = match_result['recommended_action']
        
        if 'Auto-accept' in recommendation:
            results['auto_accepted'].append(match_result)
        elif 'Manual review' in recommendation:
            results['manual_review'].append(match_result)
        else:
            results['manual_entry'].append(match_result)
        
        results['processed_medicines'].append(match_result)
    
    # Calculate statistics
    total = len(extracted_texts)
    results['statistics'] = {
        'total_processed': total,
        'auto_accepted_count': len(results['auto_accepted']),
        'manual_review_count': len(results['manual_review']),
        'manual_entry_count': len(results['manual_entry']),
        'auto_acceptance_rate': len(results['auto_accepted']) / total * 100 if total > 0 else 0
    }
    
    return results

# Test the integration
test_extracted_texts = ocr_extracted_text[:10]  # First 10 for testing
integration_results = process_extracted_medicines(test_extracted_texts, medicine_database)

print("Integration Test Results:")
print("=" * 40)
print(f"Total processed: {integration_results['statistics']['total_processed']}")
print(f"Auto-accepted: {integration_results['statistics']['auto_accepted_count']}")
print(f"Manual review needed: {integration_results['statistics']['manual_review_count']}")
print(f"Manual entry needed: {integration_results['statistics']['manual_entry_count']}")
print(f"Auto-acceptance rate: {integration_results['statistics']['auto_acceptance_rate']:.1f}%")

print("\nAuto-accepted medicines:")
for result in integration_results['auto_accepted']:
    print(f"  '{result['input_text']}' -> '{result['best_match']['medicine']}' "
          f"({result['best_match']['similarity']:.3f})")

print("\nManual review needed:")
for result in integration_results['manual_review']:
    print(f"  '{result['input_text']}' -> '{result['best_match']['medicine']}' "
          f"({result['best_match']['similarity']:.3f})")

## 12. Summary and Next Steps

### What we've learned:

1. **TF-IDF Vectorization**: Converts text into numerical vectors using character n-grams
2. **Cosine Similarity**: Measures similarity between vectors (0 = no similarity, 1 = identical)
3. **OCR Error Handling**: Preprocessing helps handle common OCR misreads
4. **Threshold Setting**: Different thresholds for automatic vs manual processing
5. **Practical Implementation**: Ready-to-use class for medicine name matching

### Key Findings:
- Character n-grams (2-4) work well for handling OCR errors
- Similarity threshold of 0.8+ provides high confidence matches
- Preprocessing significantly improves matching accuracy
- The system can handle various types of OCR errors effectively

### Next Steps:
1. **Expand Database**: Add more medicine names and variations
2. **Improve Preprocessing**: Add more OCR error patterns
3. **Machine Learning**: Train models on actual OCR data
4. **Performance Optimization**: Use sparse matrices and indexing for large databases
5. **Integration**: Connect with your existing OCR and recommendation systems

In [ ]:
# Save the matcher for future use
import pickle

# Save the trained matcher
with open('medicine_name_matcher.pkl', 'wb') as f:
    pickle.dump(matcher, f)

print("Medicine Name Matcher saved to 'medicine_name_matcher.pkl'")
print("\nTo load and use in your main system:")
print("```python")
print("import pickle")
print("with open('medicine_name_matcher.pkl', 'rb') as f:")
print("    matcher = pickle.load(f)")
print("result = matcher.find_best_match('your_ocr_text')")
print("```")

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("You now understand TF-IDF vectorization and cosine similarity")
print("for medicine name matching in OCR applications.")
print("="*60)